In [0]:
# Ranking - row_number(),rank(), dense_rank()

data =[ ('Lisa', 'Sales', 10000, 35),
          ('Evan', 'Sales', 32000, 38),
          ('Fred', 'Engineering', 21000, 28),
          ('Alex', 'Sales', 30000, 33),
          ('Tom', 'Engineering', 23000, 33),
          ('Jane', 'Marketing', 29000, 28),
          ('Jeff', 'Marketing', 35000, 38),
          ('Paul', 'Engineering', 29000, 23),
          ('Chloe', 'Engineering', 23000, 25)]

df = spark.createDataFrame(data, ['name', 'dept', 'salary', 'age'])
df.show()

from pyspark.sql.functions import *
from pyspark.sql.window import Window

w_spec = Window.partitionBy("dept").orderBy("salary")
df.withColumn("row_number", row_number().over(w_spec)) \
  .withColumn("rank", rank().over(w_spec)) \
  .withColumn("dense_rank", dense_rank().over(w_spec)) \
  .show()




In [0]:
#window + value or analytical
#lead -- next value in the window
#lag - prvious value in the window
#first value 
#last value (have to check this)

from pyspark.sql.functions import col
from pyspark.sql.window import Window
from pyspark.sql.functions import *

sch_str="txnid long,txndt string,custid long,amt float,cat string,prod string,city string,state string,spendby string"

txn_df = spark.read.csv("/Volumes/izwe48catalog/we48db/staging/txns_2025.txt",schema=sch_str)

txn_df1 = txn_df.withColumn("txndt",to_date(col("txndt"),"MM-dd-yyyy"))

w_spec = Window.partitionBy("custid").orderBy("txndt")

cust_txn_sale_df = txn_df1.withColumn("prev_amt",lag("amt").over(w_spec)) \
                          .withColumn("next_amt",lead("amt").over(w_spec)) \
                          .withColumn("first_amt",first("amt").over(w_spec)) \
                          .withColumn("last_amt",last("amt").over(w_spec))


cust_sel_df=cust_txn_sale_df.select("custid","txndt","amt","prev_amt","next_amt","first_amt","last_amt")

cust_sel_df.filter("custid=4000001").show()




In [0]:
# window + aggregation
# count,sum,avg,min,max

